# Chapter 2: GMS Fundamentals for Agent Builders

**Monograph:** *How to Build Policy-Governed AI Agents*

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Ingest a domain knowledge document into a GMS expert store on GPU.
2. Use the six GMS primitives as cognitive building blocks for an agent: `score_triple` (belief strength), `tension_energy` (contradiction), `check_holonomy` (reasoning consistency), `lookup_enm` (exact recall), `link_predict` (exploration), `query_triples` (memory retrieval).
3. Understand RuntimeLearner for continuous knowledge growth.
4. See how GMS primitives map to agentic capabilities.

## Prerequisites

- Chapter 1 notebook completed (familiarity with the governance problem).
- The domain knowledge document `governed_agent/knowledge/data_science_practices.md` is bundled with this monograph.

---
## Preamble

In [ ]:
# === Preamble (all notebooks) ===
import warnings
warnings.filterwarnings("ignore")

import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}, CUDA: {torch.cuda.is_available()}")


---
## Core Concepts

Before working with code, we define the concepts that underpin every GMS operation. The full mathematical treatment is in Appendix A.

**Knowledge triple $(h, r, t)$.** The atomic unit of structured knowledge: a **head entity** $h$, a **relation** $r$ and a **tail entity** $t$. For example, `(GradientBoosting, has_recommended_model, XGBoost)`. A GMS store is a collection of such triples together with learned geometric representations.

**Unit hypersphere $S^{m-1}$.** Every entity is embedded as a point on the surface of a unit sphere in $m$-dimensional space (typically $m=64$). The notation $S^{m-1}$ means the $(m{-}1)$-dimensional surface of this sphere. Placing entities on a sphere imposes a natural distance metric (the **geodesic** --- shortest path along the surface) and prevents embeddings from drifting to arbitrarily large magnitudes.

**Dual embeddings: v-space and u-space.** Each entity has *two* embeddings:
- **v-space** (semantic): related entities lie close together. Drives `score_triple` and `link_predict`.
- **u-space** (logical): entities that agree point in the same direction; contradictory entities point in opposite directions. Drives `tension_energy`.

**Cayley rotor $R_r$.** Each relation is a learnable *rotation matrix* on the hypersphere. Given head embedding $\tilde{v}_h$, the rotor transports it toward the tail: $R_r \tilde{v}_h \approx \tilde{v}_t$. Composing two rotors ($R_{r_2} R_{r_1}$) corresponds to chaining two reasoning steps --- this is the basis of the holonomy primitive.

**Geodesic distance.** The distance between two points on $S^{m-1}$ measured along the sphere's surface: $d(u, v) = \arccos(\langle u, v \rangle)$. A shorter geodesic means higher plausibility.

---
## The Domain Knowledge Document

The agent's "textbook" is `governed_agent/knowledge/data_science_practices.md`. This document encodes data science best practices in structured markdown -- tables and bullet lists that the GMS ingestion pipeline can parse into knowledge triples and exact numeric values.

Let us load and inspect it.

In [2]:
MD_PATH = "governed_agent/knowledge/data_science_practices.md"

with open(MD_PATH) as f:
    content = f.read()

print(f"Document loaded: {len(content):,} characters")
print(f"Lines: {len(content.splitlines())}")

# Show section headings
for line in content.splitlines():
    if line.startswith("## "):
        print(f"  {line}")

Document loaded: 10,917 characters
Lines: 226
  ## Model Selection Guidelines
  ## Preprocessing Requirements
  ## Evaluation Standards
  ## Fairness Requirements
  ## Deployment Criteria
  ## Workflow Ordering


---
## Lab 2.1 --- Build a Data Science Knowledge Store

We now ingest this document into a GMS expert store. The ingestion pipeline has four stages:

1. **Parse** -- extract triples and ENM entries from markdown tables and bullet lists
2. **Adapt** -- build index tensors mapping entity/relation names to integer IDs
3. **Train** -- embed entities on $S^{m-1}$ and learn Cayley rotors via Riemannian SGD (500 epochs on GPU)
4. **Populate** -- write exact numeric values into the ENM register with SHA-256 hashes

We use the `docgms` library which wraps the full pipeline.

In [ ]:
from knowlytix.knowledge.config import DocGMSConfig
from knowlytix.knowledge.store import GMSExpertStore
from knowlytix.knowledge.ingest import ingest_document

# Configure the store
config = DocGMSConfig(
    store_path="governance_monograph/lab_stores/ds_knowledge",
)
config.train.epochs = 500

# Create the store and ingest
store = GMSExpertStore(config, device=DEVICE)
result = ingest_document(store, MD_PATH, llm=None, config=config, device=DEVICE)

# Inspect what the agent now "knows"
stats = store.stats()
print("\n=== Knowledge Store Statistics ===")
print(f"  Entities:    {stats['entities']}")
print(f"  Relations:   {stats['relations']}")
print(f"  Triples:     {stats['triples']}")
print(f"  ENM entries: {stats['enm_entries']}")

In [4]:
# Explore the extracted triples -- what does the store know about GradientBoosting?
triples_gb = store.query_triples(head="GradientBoosting")
print(f"GradientBoosting has {len(triples_gb)} triples:")
for h, r, t in triples_gb:
    print(f"  ({h}, {r}, {t})")

print()

# What about workflow ordering?
triples_wf = store.query_triples(relation="precedes")
print(f"\nWorkflow 'precedes' triples ({len(triples_wf)}):")
for h, r, t in triples_wf:
    print(f"  {h} -> {t}")

GradientBoosting has 3 triples:
  (GradientBoosting, has_interpretable, No)
  (GradientBoosting, has_regularized, Yes (shrinkage))
  (GradientBoosting, in_section, model_selection_guidelines)


Workflow 'precedes' triples (0):


In [5]:
# Query: "What models are recommended for classification?"
print("=== link_predict: What models are recommended? ===")

# The parser created 'has_recommended_model' as the relation
# Try link prediction from a data profile
data_prof = store.fuzzy_match_entity("High-dimensional")
print(f"  Fuzzy match for 'High-dimensional': {data_prof}")

if data_prof:
    preds = store.link_predict(data_prof, "has_recommended_model", top_k=10)
    if preds:
        print(f"\n  ({data_prof}, has_recommended_model, ?) -- top 10:")
        for entity, distance in preds:
            print(f"    {entity:45s}  geodesic = {distance:.4f}")

# Also try query_triples for ground truth
print("\n=== query_triples: All recommended models ===")
rec_triples = store.query_triples(relation="has_recommended_model")
for h, r, t in rec_triples[:15]:
    score = store.score_triple(h, r, t)
    score_str = f"{score:.4f}" if score is not None else "N/A"
    print(f"  ({h}, {r}, {t})  plausibility = {score_str}")

=== link_predict: What models are recommended? ===
  Fuzzy match for 'High-dimensional': High

  (High, has_recommended_model, ?) -- top 10:
    High                                           geodesic = 0.1049
    Outliers detected above 3 sigma                geodesic = 1.2510
    LogisticRegression/Yes/Yes (L1/L2)             geodesic = 1.2546
    Cross-validation gate                          geodesic = 1.2585
    DecisionTree                                   geodesic = 1.2697
    SVM                                            geodesic = 1.2976
    General purpose, Gaussian-like features        geodesic = 1.3063
    Encoding optional for tree models              geodesic = 1.3212
    LinearRegression/Yes/No                        geodesic = 1.3227
    Mean or median imputation                      geodesic = 1.3296

=== query_triples: All recommended models ===
  (High-dimensional, many categoricals, has_recommended_model, GradientBoosting)  plausibility = 1.2330
  (High-dimensiona

In [6]:
# Query: "What does preprocessing require?"
print("=== query_triples: preprocessing requirements ===")
prep_triples = store.query_triples(head="preprocess_data")
if prep_triples:
    for h, r, t in prep_triples:
        score = store.score_triple(h, r, t)
        score_str = f"{score:.4f}" if score is not None else "N/A"
        print(f"  ({h}, {r}, {t})  plausibility = {score_str}")
else:
    # Check what entities exist related to preprocessing
    prep_match = store.fuzzy_match_entity("preprocess")
    print(f"  Fuzzy match for 'preprocess': {prep_match}")
    if prep_match:
        triples = store.query_triples(head=prep_match)
        for h, r, t in triples[:10]:
            score = store.score_triple(h, r, t)
            score_str = f"{score:.4f}" if score is not None else "N/A"
            print(f"  ({h}, {r}, {t})  plausibility = {score_str}")

=== query_triples: preprocessing requirements ===


  (preprocess_data, has_precedes, feature_select)  plausibility = 1.2758
  (preprocess_data, has_requires, load_data)  plausibility = 1.1452
  (preprocess_data, has_phase, Data)  plausibility = 1.1932


  (preprocess_data, in_section, workflow_ordering)  plausibility = 1.0314
  (preprocess_data, has_precedes, train_model)  plausibility = 0.8526
  (preprocess_data, has_requires, load_data)  plausibility = 1.1452
  (preprocess_data, has_phase, Data)  plausibility = 1.1932
  (preprocess_data, in_section, workflow_ordering)  plausibility = 1.0314


---
## Lab 2.2 --- Test the Geometric Primitives as Cognitive Tools

The six GMS primitives are not just testing tools -- they are the cognitive building blocks for a governed agent. Each primitive maps to an agentic capability:

| Primitive | Geometric Meaning | Agentic Use |
|---|---|---|
| `score_triple(h, r, t)` | Geodesic $d(R_r \tilde{v}_h, \tilde{v}_t)$ | **Belief strength** -- how plausible is this fact? |
| `tension_energy(a, b)` | Clifford $\sqrt{2 - 2\langle u_a, u_b\rangle}$ | **Contradiction detection** -- do these facts conflict? |
| `check_holonomy(path, direct)` | Path composition defect | **Reasoning consistency** -- is this chain of inference valid? |
| `lookup_enm(cat, id)` | SHA-256 verified IEEE 754 | **Exact numeric recall** -- what was the exact threshold? |
| `link_predict(h, r, k)` | Beam search on manifold | **Exploration** -- what models work for this data type? |
| `query_triples(h, r, t)` | Graph pattern match | **Memory retrieval** -- what do we know about this entity? |

### Primitive 1: `score_triple` -- Belief Strength

The agent uses `score_triple` to judge how plausible a claim is. True facts from the document should score low (close on the manifold); fabricated facts should score high.

In [7]:
import statistics

# Score true triples from the document
print("=== Known-True Triples (from the document) ===")

# We need to work with actual entities and relations in the store.
# Let us first find some entity names to use.
sample_entities = list(store.adapter.entity_to_idx.keys())[:20]
print(f"Sample entities: {sample_entities}")

# Check which relations exist
sample_rels = sorted(store.adapter.relation_to_idx.keys())[:20]
print(f"\nSample relations: {sample_rels}")

=== Known-True Triples (from the document) ===
Sample entities: ['(none)', '0 to 1', '0.15', '0.25', '0.35', '0.5', '0.55', '0.6', '0.65', '0.7', '0.75', '0.8', '0.85', '1.0', '2.0', '3.0', '4.0', '5.0', '6.0', 'Above 0.05']

Sample relations: ['fails', 'has_affected_models', 'has_applicability', 'has_attribute', 'has_condition', 'has_description', 'has_gate_type', 'has_interpretable', 'has_interpretation', 'has_maximum_rmse_ratio', 'has_minimum_auc', 'has_minimum_f1', 'has_minimum_precision', 'has_minimum_r-squared', 'has_minimum_recall', 'has_missing_pattern', 'has_must_test', 'has_order', 'has_phase', 'has_precedes']


In [8]:
# Score true triples vs fabricated triples
# We pick triples directly from the stored graph (ground truth)
all_triples = list(store.doc_graph.triples)
print(f"Total triples in store: {len(all_triples)}")

# Select 10 true triples with entities that exist as separate embeddings
true_triples = []
for h, r, t in all_triples:
    if (store.entity_exists(h) and store.entity_exists(t) 
        and store.relation_exists(r)):
        score = store.score_triple(h, r, t)
        if score is not None:
            true_triples.append((h, r, t, score))
    if len(true_triples) >= 10:
        break

print("\n=== Known-True Triples ===")
true_scores = []
for h, r, t, score in true_triples:
    true_scores.append(score)
    print(f"  ({h}, {r}, {t}):  d = {score:.4f}")

if true_scores:
    print(f"\n  Mean: {statistics.mean(true_scores):.4f}  "
          f"Std: {statistics.stdev(true_scores):.4f}")

Total triples in store: 370

=== Known-True Triples ===
  (High-dimensional, many categoricals, has_recommended_model, GradientBoosting):  d = 1.2330
  (High-dimensional, many categoricals, has_rationale, Handles categorical features natively, built-in regularization):  d = 1.4941
  (High-dimensional, many categoricals, in_section, model_selection_guidelines):  d = 1.1672
  (High-dimensional, many categoricals, has_recommended_model, XGBoost):  d = 1.3501
  (High-dimensional, many categoricals, has_rationale, Handles categorical features natively, built-in regularization):  d = 1.4941
  (High-dimensional, many categoricals, in_section, model_selection_guidelines):  d = 1.1672
  (High-dimensional, many categoricals, has_recommended_model, RandomForest):  d = 1.2479
  (High-dimensional, many categoricals, has_rationale, Ensemble averaging reduces variance, handles mixed types):  d = 1.5018
  (High-dimensional, many categoricals, in_section, model_selection_guidelines):  d = 1.1672
  (Sma

In [9]:
# Now fabricate triples by swapping tail entities
import random
random.seed(42)

entity_names = list(store.adapter.entity_to_idx.keys())

fab_triples = []
for h, r, t, _ in true_triples:
    # Pick a random tail that is different from the true tail
    wrong_t = random.choice([e for e in entity_names if e != t])
    score = store.score_triple(h, r, wrong_t)
    if score is not None:
        fab_triples.append((h, r, wrong_t, score))

print("=== Fabricated Triples (wrong tail entity) ===")
fab_scores = []
for h, r, t, score in fab_triples:
    fab_scores.append(score)
    print(f"  ({h}, {r}, {t}):  d = {score:.4f}")

if fab_scores:
    print(f"\n  Mean: {statistics.mean(fab_scores):.4f}  "
          f"Std: {statistics.stdev(fab_scores):.4f}")

    print("\n=== Comparison ===")
    print(f"  True triples mean:       {statistics.mean(true_scores):.4f}")
    print(f"  Fabricated triples mean: {statistics.mean(fab_scores):.4f}")
    sep = statistics.mean(fab_scores) - statistics.mean(true_scores)
    print(f"  Separation:              {sep:+.4f}")
    print("\n  Fabricated triples score higher (further from ground truth).")

=== Fabricated Triples (wrong tail entity) ===
  (High-dimensional, many categoricals, has_recommended_model, Optimal under linearity assumption):  d = 1.2534
  (High-dimensional, many categoricals, has_rationale, Advisory gates failed only):  d = 1.6204
  (High-dimensional, many categoricals, in_section, 0.55):  d = 1.3970
  (High-dimensional, many categoricals, has_recommended_model, StandardScaler):  d = 1.3444
  (High-dimensional, many categoricals, has_rationale, DecisionTree/Yes (depth<=3)/No):  d = 1.4799
  (High-dimensional, many categoricals, in_section, Credit and lending models):  d = 1.4570
  (High-dimensional, many categoricals, has_recommended_model, Class weight adjustment, bootstrap sampling):  d = 1.8684
  (High-dimensional, many categoricals, has_rationale, All must-pass except one should-pass):  d = 1.4382
  (High-dimensional, many categoricals, in_section, Small dataset below 1000 rows):  d = 1.5319
  (Small dataset below 1000 rows, has_recommended_model, Adverse Im

> **v-space vs u-space:** `score_triple` (Primitive 1) works in v-space (semantic proximity). `tension_energy` works in u-space (logical alignment). Two entities can be semantically close in v-space (both are ML models) yet logically opposed in u-space (one is the best choice, the other is the worst). This dual representation is what allows GMS to distinguish "related" from "agrees/contradicts."

### Primitive 2: `tension_energy` -- Contradiction Detection

Tension energy operates in **u-space** (the logical embedding). It measures how strongly two entities agree, are irrelevant to each other or contradict each other:

$$E(u_a, u_b) = \sqrt{2 - 2\langle u_a, u_b\rangle}$$

| Energy Range | Meaning |
|---|---|
| $E \approx 0$ | Agreement (vectors aligned) |
| $E \approx \sqrt{2} \approx 1.414$ | Irrelevance (vectors orthogonal) |
| $E \approx 2$ | Contradiction (vectors antipodal) |

An agent uses tension energy to decide: "Does this new fact contradict what I already know?"

In [10]:
import math

# Related concepts -- should have lower tension (more agreement)
print("=== Tension Energy: Related Concepts ===")
related_pairs = []
for a, b in [("GradientBoosting", "XGBoost"),
             ("LogisticRegression", "LinearRegression"),
             ("RandomForest", "GradientBoosting"),
             ("StandardScaler", "MinMaxScaler")]:
    t = store.tension_energy(a, b)
    if t is not None:
        related_pairs.append((a, b, t))
        print(f"  E({a}, {b}) = {t:.4f}")
    else:
        print(f"  E({a}, {b}) = None (entity not found)")

# Semantically opposed concepts -- should have higher tension
print("\n=== Tension Energy: Opposed Concepts ===")
opposed_pairs = []
for a, b in [("High", "Low"),
             ("Critical", "Low"),
             ("overfitting", "underfitting")]:
    t = store.tension_energy(a, b)
    if t is not None:
        opposed_pairs.append((a, b, t))
        print(f"  E({a}, {b}) = {t:.4f}")
    else:
        # Try fuzzy match
        a_m = store.fuzzy_match_entity(a)
        b_m = store.fuzzy_match_entity(b)
        print(f"  E({a}, {b}) = None  (fuzzy: {a_m}, {b_m})")

# Cross-domain concepts -- should be near irrelevance boundary
print("\n=== Tension Energy: Unrelated Concepts ===")
for a, b in [("GradientBoosting", "0.85"),
             ("StandardScaler", "High-risk (credit, lending)")]:
    t = store.tension_energy(a, b)
    if t is not None:
        print(f"  E({a}, {b}) = {t:.4f}")
    else:
        a_m = store.fuzzy_match_entity(a)
        b_m = store.fuzzy_match_entity(b)
        if a_m and b_m:
            t = store.tension_energy(a_m, b_m)
            if t is not None:
                print(f"  E({a_m}, {b_m}) = {t:.4f}")

print(f"\n  Reference: sqrt(2) = {math.sqrt(2):.4f} (irrelevance boundary)")

=== Tension Energy: Related Concepts ===


  E(GradientBoosting, XGBoost) = 1.4308
  E(LogisticRegression, LinearRegression) = 1.4416
  E(RandomForest, GradientBoosting) = 1.4948
  E(StandardScaler, MinMaxScaler) = 1.4378

=== Tension Energy: Opposed Concepts ===
  E(High, Low) = 1.4324
  E(Critical, Low) = 1.3349
  E(overfitting, underfitting) = None  (fuzzy: L2 regularization prevents overfitting, None)

=== Tension Energy: Unrelated Concepts ===
  E(GradientBoosting, 0.85) = 1.4413
  E(StandardScaler, High-risk (credit, lending)) = 1.4717

  Reference: sqrt(2) = 1.4142 (irrelevance boundary)


### Primitive 3: `check_holonomy` -- Reasoning Consistency

When we compose two relation rotors -- for example, "requires" followed by "precedes" -- we get a path rotor $R_{\text{path}} = R_2 R_1$. The **holonomy defect** compares this composed path to a single direct relation rotor:

$$H = \|R_{\text{path}} R_{\text{direct}}^{-1} - I\|_F$$

If $H \approx 0$, the two paths are geometrically consistent. An agent uses this to verify that multi-step reasoning chains are internally consistent -- for example, whether the workflow ordering "load_data requires (nothing), preprocess_data requires load_data" is consistent with the direct "precedes" relation.

In [11]:
# Check holonomy defects for workflow-related relation paths
print("=== Holonomy Defect: Workflow Consistency ===")

# The store has: has_precedes, has_requires, has_phase, has_order
workflow_rels = [r for r in store.adapter.relation_to_idx.keys()
                 if any(kw in r for kw in ["precedes", "requires", "phase", "order"])]
print(f"Workflow-related relations: {workflow_rels}")

# Test: does (has_requires + has_precedes) form a consistent path vs has_phase?
holonomy_cases = [
    (["has_requires", "has_precedes"], "has_phase",
     "requires + precedes vs phase"),
    (["has_phase", "has_order"], "has_precedes",
     "phase + order vs precedes"),
    (["has_precedes", "has_phase"], "has_requires",
     "precedes + phase vs requires"),
]

for path, direct, description in holonomy_cases:
    h = store.check_holonomy(path, direct)
    if h is not None:
        consistent = "YES" if h < 0.5 else "NO"
        print(f"\n  {description}")
        print(f"    path = {path} -> direct = {direct}")
        print(f"    holonomy defect = {h:.4f}  consistent: {consistent}")

# Compare with unrelated relation combinations
print("\n=== Holonomy Defect: Unrelated Relations (baseline) ===")
unrelated_cases = [
    (["has_recommended_model", "has_rationale"], "in_section",
     "recommended_model + rationale vs in_section"),
    (["has_threshold", "has_regulation"], "has_gate_type",
     "threshold + regulation vs gate_type"),
]

for path, direct, description in unrelated_cases:
    h = store.check_holonomy(path, direct)
    if h is not None:
        print(f"  {description}")
        print(f"    holonomy defect = {h:.4f}")

=== Holonomy Defect: Workflow Consistency ===
Workflow-related relations: ['has_order', 'has_phase', 'has_precedes', 'has_requires']

  requires + precedes vs phase
    path = ['has_requires', 'has_precedes'] -> direct = has_phase
    holonomy defect = 1.2499  consistent: NO

  phase + order vs precedes
    path = ['has_phase', 'has_order'] -> direct = has_precedes
    holonomy defect = 1.2444  consistent: NO

  precedes + phase vs requires
    path = ['has_precedes', 'has_phase'] -> direct = has_requires
    holonomy defect = 1.2973  consistent: NO

=== Holonomy Defect: Unrelated Relations (baseline) ===
  recommended_model + rationale vs in_section
    holonomy defect = 1.3311
  threshold + regulation vs gate_type
    holonomy defect = 1.2892


### Primitive 4: `lookup_enm` -- Exact Numeric Recall

Some facts are inherently numeric: "the minimum AUC for high-risk models is 0.85." Embedding a number as a point on a sphere would lose precision. For these facts GMS uses a separate **Exact Numerical Memory (ENM)** register with SHA-256 integrity hashes.

An agent uses ENM for lossless recall of thresholds, metrics and other numeric values.

In [12]:
# Explore ENM entries -- what numeric values are stored?
print("=== ENM Register Contents ===")
all_enm_keys = list(store.enm.keys()) if store.enm else []
print(f"Total ENM entries: {len(all_enm_keys)}")

# Show first 20 entries
print("\nFirst 20 ENM entries:")
for key in all_enm_keys[:20]:
    val = store.enm.read_exact(key)
    if val is not None:
        v = float(val.item() if hasattr(val, 'item') else val)
        print(f"  ({key.type}, {key.id}) = {v}")

# Look up specific threshold values
print("\n=== Threshold Lookups ===")
# Search for entries containing common threshold-related terms
for key in all_enm_keys:
    key_str = f"{key.type}::{key.id}".lower()
    if any(term in key_str for term in ["auc", "f1", "0.8", "0.7", "0.6"]):
        val = store.enm.read_exact(key)
        if val is not None:
            v = float(val.item() if hasattr(val, 'item') else val)
            print(f"  ({key.type}, {key.id}) = {v}")

=== ENM Register Contents ===
Total ENM entries: 24

First 20 ENM entries:
  (evaluation_standards, High-risk (credit, lending)/Fairness, robustness, explanation, cross-validation/Minimum AUC) = 0.85
  (evaluation_standards, High-risk (credit, lending)/Fairness, robustness, explanation, cross-validation/Minimum F1) = 0.8
  (evaluation_standards, High-risk (credit, lending)/Fairness, robustness, explanation, cross-validation/Minimum Precision) = 0.75
  (evaluation_standards, High-risk (credit, lending)/Fairness, robustness, explanation, cross-validation/Minimum Recall) = 0.75
  (evaluation_standards, Medium-risk (marketing, operations)/Accuracy, explanation, cross-validation/Minimum AUC) = 0.75
  (evaluation_standards, Medium-risk (marketing, operations)/Accuracy, explanation, cross-validation/Minimum F1) = 0.7
  (evaluation_standards, Medium-risk (marketing, operations)/Accuracy, explanation, cross-validation/Minimum Precision) = 0.65
  (evaluation_standards, Medium-risk (marketing, op

In [13]:
# Verify SHA-256 integrity of all ENM entries
print("=== ENM Integrity Verification ===")
n_verified = 0
n_failed = 0
for k in all_enm_keys:
    ok = store.enm.verify_integrity(k)
    if ok:
        n_verified += 1
    else:
        n_failed += 1

print(f"  {n_verified}/{len(all_enm_keys)} entries passed SHA-256 verification")
print(f"  {n_failed}/{len(all_enm_keys)} entries failed")

# Show the hash for one entry if available
if all_enm_keys:
    sample_key = all_enm_keys[0]
    entry = store.enm._store[sample_key]
    print(f"\n  Example: ({sample_key.type}, {sample_key.id})")
    print(f"    Value:     {float(entry.value)}")
    print(f"    SHA-256:   {entry.hash.hex()[:32]}...")
    print(f"    Integrity: {store.enm.verify_integrity(sample_key)}")

=== ENM Integrity Verification ===
  24/24 entries passed SHA-256 verification
  0/24 entries failed

  Example: (evaluation_standards, High-risk (credit, lending)/Fairness, robustness, explanation, cross-validation/Minimum AUC)
    Value:     0.85
    SHA-256:   37249e03321df964601ce1432b5a1b66...
    Integrity: True


### Primitive 5: `link_predict` -- Exploration

`link_predict(h, r, top_k)` answers: "Given head entity $h$ and relation $r$, which tail entities are most plausible?" It computes $d(h, r, t)$ for every entity $t$ in the vocabulary and returns the top-$k$ with the lowest geodesic distance.

An agent uses link prediction to explore: "What models handle categorical features?" or "What preprocessing is needed for SVM?"

In [14]:
# Link prediction: explore the knowledge store
print("=== link_predict exploration ===\n")

gb_match = store.fuzzy_match_entity("GradientBoosting")
lr_match = store.fuzzy_match_entity("LogisticRegression")
print(f"GradientBoosting match: {gb_match}")
print(f"LogisticRegression match: {lr_match}")

# Which data profiles recommend GradientBoosting?
if gb_match:
    for rel in ["has_recommended_model", "has_rationale", "in_section"]:
        preds = store.link_predict(gb_match, rel, top_k=5)
        if preds:
            print(f"\n  ({gb_match}, {rel}, ?) -- top 5:")
            for entity, score in preds:
                print(f"    {entity:45s}  geodesic = {score:.4f}")

# What does LogisticRegression relate to?
if lr_match:
    for rel in ["has_recommended_model", "has_interpretable"]:
        preds = store.link_predict(lr_match, rel, top_k=5)
        if preds:
            print(f"\n  ({lr_match}, {rel}, ?) -- top 5:")
            for entity, score in preds:
                print(f"    {entity:45s}  geodesic = {score:.4f}")

=== link_predict exploration ===

GradientBoosting match: GradientBoosting
LogisticRegression match: LogisticRegression

  (GradientBoosting, has_recommended_model, ?) -- top 5:
    GradientBoosting                               geodesic = 0.1294
    Model beats baseline                           geodesic = 1.1234
    High-dimensional, many categoricals            geodesic = 1.2467
    SVM/No/Yes (C parameter)                       geodesic = 1.2558
    Yes (shrinkage)                                geodesic = 1.2609



  (GradientBoosting, has_rationale, ?) -- top 5:
    GradientBoosting                               geodesic = 0.1026
    Model beats baseline                           geodesic = 1.1508
    High-dimensional, many categoricals            geodesic = 1.2234
    SVM/No/Yes (C parameter)                       geodesic = 1.2316
    Yes (shrinkage)                                geodesic = 1.2443

  (GradientBoosting, in_section, ?) -- top 5:
    GradientBoosting                               geodesic = 0.1434
    Model beats baseline                           geodesic = 1.1554
    model_selection_guidelines                     geodesic = 1.2124
    High-dimensional, many categoricals            geodesic = 1.2194
    Yes (shrinkage)                                geodesic = 1.2278

  (LogisticRegression, has_recommended_model, ?) -- top 5:
    LogisticRegression                             geodesic = 0.1191
    Need interpretability                          geodesic = 1.2028
    GradientBoo

### Primitive 6: `query_triples` -- Memory Retrieval

`query_triples` performs pattern matching on the document graph. Unlike the other primitives, this is not a learned geometric operation -- it simply filters stored triples by head, relation and/or tail.

An agent uses this for structured recall: "What do I know about GradientBoosting?" or "What steps require preprocessing?"

In [15]:
# Memory retrieval via query_triples
print("=== query_triples: Everything about GradientBoosting ===")
gb_match = store.fuzzy_match_entity("GradientBoosting")
if gb_match:
    triples = store.query_triples(head=gb_match)
    for h, r, t in triples:
        print(f"  ({h}, {r}, {t})")

# Query by relation: workflow dependencies
print("\n=== query_triples(relation='has_requires') ===")
req_triples = store.query_triples(relation="has_requires")
for h, r, t in req_triples:
    print(f"  {h} requires {t}")

# Query by relation: step ordering
print("\n=== query_triples(relation='has_precedes') ===")
prec_triples = store.query_triples(relation="has_precedes")
for h, r, t in prec_triples:
    print(f"  {h} -> {t}")

# What leads to deployment?
print("\n=== query_triples: deployment path ===")
deploy_match = store.fuzzy_match_entity("deploy")
if deploy_match:
    triples = store.query_triples(tail=deploy_match)
    for h, r, t in triples:
        print(f"  ({h}, {r}, {t})")
    triples = store.query_triples(head=deploy_match)
    for h, r, t in triples:
        print(f"  ({h}, {r}, {t})")

=== query_triples: Everything about GradientBoosting ===
  (GradientBoosting, has_interpretable, No)
  (GradientBoosting, has_regularized, Yes (shrinkage))
  (GradientBoosting, in_section, model_selection_guidelines)

=== query_triples(relation='has_requires') ===
  load_data requires (none)
  preprocess_data requires load_data
  preprocess_data requires load_data
  feature_select requires preprocess_data
  train_model requires preprocess_data
  train_model requires preprocess_data
  diagnose_accuracy requires train_model
  diagnose_accuracy requires train_model
  diagnose_robustness requires diagnose_accuracy
  diagnose_fairness requires train_model
  explain_importance requires train_model
  explain_importance requires train_model
  compare_models requires diagnose_accuracy
  release_gate requires compare_models

=== query_triples(relation='has_precedes') ===
  load_data -> preprocess_data
  preprocess_data -> feature_select
  preprocess_data -> train_model
  feature_select -> train_

---
## RuntimeLearner: Continuous Knowledge Growth

An agent is not static -- it learns from experience. The `RuntimeLearner` enables the GMS store to grow at runtime via a two-phase protocol:

- **Phase 1 (synchronous):** Verify the new triple against the contradiction gate, add it to the document graph immediately, expand the adapter indices and write any numeric values to ENM. Graph queries work immediately.
- **Phase 2 (asynchronous):** Expand embedding tables, add new Cayley rotors, run Riemannian SGD on the new entities in a background thread. Geometric scoring becomes available once training completes.

This lets the agent learn new facts without retraining the entire model.

In [ ]:
from knowlytix.knowledge.learn import RuntimeLearner
from knowlytix.knowledge.extract import ExtractedTriple

learner = RuntimeLearner(store, config.learn)

# Add a new triple: the agent just learned something
new_triples = [
    ExtractedTriple(
        head="CatBoost",
        relation="is_type",
        tail="GradientBoosting",
    ),
    ExtractedTriple(
        head="CatBoost",
        relation="handles",
        tail="categorical_features",
    ),
]

# Before learning: CatBoost does not exist
print(f"CatBoost exists before learning: {store.entity_exists('CatBoost')}")

# Learn the new triples
result = learner.learn_triples(new_triples, DEVICE)
print("\nLearn result:")
print(f"  Accepted: {result.accepted}")
print(f"  Reason: {result.reason}")
print(f"  New entities: {result.new_entities}")
print(f"  Triples added: {result.n_triples_added}")

# Phase 1 is synchronous -- graph queries work immediately
print(f"\nCatBoost exists after learning: {store.entity_exists('CatBoost')}")
cb_triples = store.query_triples(head="CatBoost")
print(f"CatBoost triples: {len(cb_triples)}")
for h, r, t in cb_triples:
    print(f"  ({h}, {r}, {t})")

# Flush background training (Phase 2)
learner.flush_training()
learner.wait_for_training(timeout=60)
print(f"\nTraining status: {learner.training_status()}")

In [17]:
# After Phase 2 training, geometric scoring works for the new entity
score = store.score_triple("CatBoost", "is_type", "GradientBoosting")
if score is not None:
    print(f"score_triple(CatBoost, is_type, GradientBoosting) = {score:.4f}")
else:
    print("Geometric scoring not yet available (training may still be running)")

# Updated stats
stats = store.stats()
print("\n=== Updated Knowledge Store Statistics ===")
print(f"  Entities:    {stats['entities']}")
print(f"  Relations:   {stats['relations']}")
print(f"  Triples:     {stats['triples']}")
print(f"  ENM entries: {stats['enm_entries']}")

score_triple(CatBoost, is_type, GradientBoosting) = 0.0281

=== Updated Knowledge Store Statistics ===
  Entities:    246
  Relations:   40
  Triples:     372
  ENM entries: 24


---
## Exercises

### Exercise 2.1

Ingest a second document (create a short `adult_income_domain.md` with 5-10 facts about the Adult Income dataset). Verify that RuntimeLearner's contradiction gate rejects any triple that conflicts with the existing best-practices knowledge (e.g., a triple claiming "LogisticRegression handles categorical_features" when the store already knows it does not).

In [18]:
# === Exercise 2.1: Your code here ===


### Exercise 2.2

Use `query_triples` and `link_predict` to find all reasoning paths from "missing values" to "model accuracy" concepts (max 4 hops). Interpret the paths -- what does GMS know about how missing values affect accuracy?

In [19]:
# === Exercise 2.2: Your code here ===


### Exercise 2.3

Add 5 deliberately wrong triples (e.g., "LinearRegression best_for image_classification", "SVM handles missing_values"). Show that `tension_energy` is above 1.5 for at least some of these entity pairs, and that `RuntimeLearner.learn_triples()` rejects them at the contradiction gate.

In [20]:
# === Exercise 2.3: Your code here ===


---
## Summary

In this chapter we built the agent's domain knowledge store -- its "textbook" of data science best practices -- and explored the six geometric primitives that serve as cognitive building blocks:

1. **`score_triple`** measures geodesic distance on $S^{m-1}$ -- the agent's belief strength about a fact. True facts from the document score lower than fabricated ones.

2. **`tension_energy`** uses u-space embeddings to detect contradictions. The energy spectrum ranges from 0 (agreement) through $\sqrt{2}$ (irrelevance) to 2 (contradiction).

3. **`check_holonomy`** compares composed path rotors to direct rotors, enabling the agent to verify that multi-step reasoning is internally consistent.

4. **`lookup_enm`** provides lossless numeric recall with SHA-256 integrity -- the agent retrieves exact thresholds and metric values without embedding approximation.

5. **`link_predict`** performs beam search on the manifold for exploration -- the agent discovers related concepts and candidate answers.

6. **`query_triples`** provides exact structural matching for memory retrieval -- the agent recalls specific facts from its knowledge graph.

We also demonstrated **RuntimeLearner** for continuous knowledge growth: the agent can learn new facts at runtime without full retraining, with a contradiction gate that rejects conflicting information.

In the next chapter we build the 15 sklearn tools that give the agent the ability to act on this knowledge.